In [1]:
import numpy as np

In [2]:
def mean(values: list[float]):
    return sum(values)/len(values)

In [3]:
def covariance_pair(x: list[float], y: list[float]) -> float:
    n = len(x)
    
    if n != len(y):
        raise ValueError("Features must have the same number of observations.")
    
    x_bar = mean(x)
    y_bar = mean(y)
    num = 0
    
    for i in range(n):
        num += (x[i] - x_bar)*(y[i] - y_bar)
    return num/(n-1)

In [4]:
def print_deviation_table(data, means):
    n_features = len(data)
    n_samples = len(data[0])

    print("\n\nDeviation Table")
    print("-" * (12 + n_features * 20))

    header = f"{'i':<5}"

    for j in range(n_features):
        header += f"{'X' + str(j + 1):>10}"
        header += f"{'X' + str(j + 1) + f'-X{str(j + 1)}_mean':>12}"

    print(header)
    print("-" * (12 + n_features * 20))

    deviations = []

    for i in range(n_samples):
        row = f"{i + 1:<5}"
        dev_row = []

        for j in range(n_features):
            value = data[j][i]
            deviation = value - means[j]

            dev_row.append(deviation)

            row += f"{value:10.3f}"
            row += f"{deviation:12.3f}"

        deviations.append(dev_row)
        print(row)

    return deviations

In [5]:
def print_product_table(deviations):
    n_samples = len(deviations)
    n_features = len(deviations[0])

    pairs = []

    for j in range(n_features):
        for k in range(j, n_features):
            pairs.append((j, k))

    print("\n\nProducts of Deviations")
    print("-" * (5 + len(pairs) * 15))

    print("Each column represents (Xi - Xi_mean) × (Xj - Xj_mean)\n")

    print(f"{'i':<5}", end="")

    for j, k in pairs:
        print(f"{'X' + str(j + 1) + '×X' + str(k + 1):>15}", end="")

    print()
    print("-" * (5 + len(pairs) * 15))

    for i in range(n_samples):
        print(f"{i + 1:<5}", end="")

        for j, k in pairs:
            product = deviations[i][j] * deviations[i][k]
            print(f"{product:15.3f}", end="")

        print()

In [6]:
def covariance_matrix(data: list[list[float]]) -> list[list[float]]:
    n_features = len(data)
    
    lengths = {len(col) for col in data}
    if len(lengths) > 1:
        raise ValueError(f"All features must have equal samples. Got lengths: {lengths}")
    
    means = [mean(feature) for feature in data]

    print("\nMeans:")
    for i, m in enumerate(means):
        print(f"Feature {i + 1} mean = {m:.3f}")

    deviations = print_deviation_table(data, means)
    print_product_table(deviations)
    
    cov_matrix = [[0.0] * n_features for _ in range(n_features)]
    for i in range(n_features):
        for j in range(i,n_features):
            cov_val = covariance_pair(data[i],data[j])
            cov_matrix[i][j] = cov_val
            cov_matrix[j][i] = cov_val
            
    return cov_matrix

In [7]:
def print_matrix(matrix: list[list[float]]) -> None:
    print("\nCovariance Matrix\n")
    for row in matrix:
        print("  " + "  ".join(f"{val:10.3f}" for val in row))

In [8]:
def parse_floats(raw: str) -> list[float]:
    raw = raw.replace(",", " ")
    return [float(x) for x in raw.split()]
 
 
def get_data() -> tuple[list[list[float]], list[str]]:
    while True:
        try:
            n_features = int(input("Enter no. of features: ").strip())
            if n_features < 2:
                print("Need at least 2 features to compute covariance.")
                continue
            break
        except ValueError:
            print("Please enter a valid integer.")
 
    while True:
        try:
            n_samples = int(input("Enter no. of samples: ").strip())
            if n_samples < 2:
                print("Need at least 2 samples.")
                continue
            break
        except ValueError:
            print("Please enter a valid integer.")
 
    print(f"\nEnter {n_samples} values for each feature.")
    print("  (space- or comma-separated on one line, e.g.: 1 2 3  or  1,2,3)\n")
 
    data = []
 
    for i in range(n_features):
 
        while True:
            raw = input(f"Values for feature {i+1}: ").strip()
            try:
                values = parse_floats(raw)
                if len(values) != n_samples:
                    print(f"Expected {n_samples} values, got {len(values)}. Try again.")
                    continue
                data.append(values)
                break
            except ValueError:
                print("Invalid input. Use numbers separated by spaces or commas.")
 
    return data

In [9]:
if __name__ == "__main__":
    
    data = get_data()
    
    cov_matrix = covariance_matrix(data)
    print_matrix(cov_matrix)

Enter no. of features: 3
Enter no. of samples: 8

Enter 8 values for each feature.
  (space- or comma-separated on one line, e.g.: 1 2 3  or  1,2,3)

Values for feature 1: 12.1,13.2,15.6,17.2,18.8,10.3,11.7,16.4
Values for feature 2: 48,59,32,18,41,32,31,30
Values for feature 3: 101,171,112,132,140,112,151,96

Means:
Feature 1 mean = 14.412
Feature 2 mean = 36.375
Feature 3 mean = 126.875


Deviation Table
------------------------------------------------------------------------
i            X1  X1-X1_mean        X2  X2-X2_mean        X3  X3-X3_mean
------------------------------------------------------------------------
1        12.100      -2.312    48.000      11.625   101.000     -25.875
2        13.200      -1.212    59.000      22.625   171.000      44.125
3        15.600       1.188    32.000      -4.375   112.000     -14.875
4        17.200       2.788    18.000     -18.375   132.000       5.125
5        18.800       4.388    41.000       4.625   140.000      13.125
6        10.